In [9]:
#as we know that the llm have limited context window so we can't provide all the previous messages to the llm every time so for that we use short term memory which only keeps the recent messages in context window and rest of the messages are stored in database or file system or any other persistent storage. So here we will see how to implement short term memory using langgraph.
#In this we will only keep the last n messages and remvove the rest of the history 



In [10]:
from langgraph.graph import StateGraph,START,END,MessagesState 
from langgraph.checkpoint.memory import InMemorySaver 
from langgraph.checkpoint.sqlite import SqliteSaver

from langchain_core.messages import BaseMessage,HumanMessage,AIMessage 
from langchain_core.messages import RemoveMessage 

In [11]:
from dotenv import load_dotenv 
load_dotenv()
import sqlite3

In [12]:
from langchain_groq import ChatGroq
llm=ChatGroq(model='Llama-3.3-70b-Versatile')


In [13]:
conn = sqlite3.connect(database="demo.db", check_same_thread=False)
checkpointer = SqliteSaver(conn=conn)

In [14]:
#lets design the graph for chatting with the llm 

graph=StateGraph(MessagesState)


In [15]:
def chat(state: MessagesState):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

def delete_old_messages(state: MessagesState):
    msgs = state["messages"]

    # if more than 10 messages, delete the earliest 6
    if len(msgs) > 10:
        to_remove = msgs[:6]
        return {"messages": [RemoveMessage(id=m.id) for m in to_remove]}

    return {}

In [16]:
graph.add_node('chat',chat) 
graph.add_node('delete_old_messages',delete_old_messages)

graph.add_edge(START,'chat') 
graph.add_edge('chat','delete_old_messages')
graph.add_edge('delete_old_messages',END) #it will run after the each response 


In [17]:
workflow=graph.compile(checkpointer=checkpointer)

In [18]:
config={'configurable':{'thread_id':'32'}} 

In [19]:
# # initial_state={'messages': [HumanMessage(content="Hi, My name is aashish")]}
# # initial_state={'messages': [HumanMessage(content="What is my name")]}
# # initial_state={'messages':[HumanMessage(content='can you tell me a joke')]}
# initial_state={'messages': [HumanMessage(content="write an paragraph about Elon Musk")]}

In [21]:
workflow.invoke({"messages": [{"role": "user", "content": "Hi, I'm Aashish"}]}, config)
workflow.invoke({"messages": [{"role": "user", "content": "Tell me about LangGraph"}]}, config)
workflow.invoke({"messages": [{"role": "user", "content": "Now explain checkpointers"}]}, config)
workflow.invoke({"messages": [{"role": "user", "content": "What is Langchain"}]}, config)
workflow.invoke({"messages": [{"role": "user", "content": "What is Quantum Mechanics"}]}, config)
workflow.invoke({"messages": [{"role": "user", "content": "What is Gen AI"}]}, config)
workflow.invoke({"messages": [{"role": "user", "content": "What is my name"}]}, config)

{'messages': [HumanMessage(content='What is Quantum Mechanics', additional_kwargs={}, response_metadata={}, id='c4b9fc18-5d5c-4474-a85f-2933b0d4bbd3'),
  AIMessage(content='Quantum Mechanics (QM) is a fundamental theory in physics that describes the behavior of matter and energy at the smallest scales, such as atoms and subatomic particles. It is a branch of physics that deals with the study of the physical properties of matter and energy at the atomic and subatomic level.\n\n**Key Principles of Quantum Mechanics:**\n\n1. **Wave-Particle Duality**: Quantum objects, such as electrons, can exhibit both wave-like and particle-like behavior depending on how they are observed.\n2. **Uncertainty Principle**: It is impossible to know certain properties of a quantum object, such as its position and momentum, simultaneously with infinite precision.\n3. **Superposition**: Quantum objects can exist in multiple states simultaneously, which is known as a superposition of states.\n4. **Entanglement*

In [ ]:
#This is simple short term memory which remains for the current execution only

In [ ]:
#if we wants that it doesn't gets deleted after the execution we have to use the dbms persistence